# HGT Final Training

1. Uses 5-fold cross-validation.
2. Uses Val AUC as the primary early-stopping criterion.
3. Adds validation-loss protection to prevent late-stage overfitting.
4. Separately logs Train BCE, Validation BCE, Total Train Loss, Validation AUC, and Validation AP.
5. Saves the best checkpoint for each fold.
6. Additionally outputs a fold summary, JSON results, learning curves, and brand embedding collapse metrics.


## 0. Imports & Config

In [ ]:
import math, random, warnings, json, gc, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.nn import HGTConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings('ignore')

# CONFIG: main settings for the official final training run
DATA_PATH   = 'YOUR_GRAPH_OUTPUT_FILE.pt'
OPTION = 'B'
K_FOLDS     = 5
MAX_EPOCHS  = 120
PATIENCE    = 15

# Early stopping:
# Primarily based on Val AUC; but if Val BCE later becomes much worse than the best Val BCE, stop early.
VAL_AUC_MIN_DELTA = 0.001
VAL_LOSS_PROTECTION_EPOCH = 80
VAL_LOSS_PROTECTION_DELTA = 0.15

SEED           = 42
DISJOINT_RATIO = 0.10
INDUSTRY_DIM   = 11

# Output directory
OUTPUT_DIR = f'hgt_final_option{OPTION}_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# optimizer parameters
_PARAMS = {
    'A': dict(hidden_dim=128, out_dim=128, num_heads=2, num_layers=2,
              dropout=0.5482, lr=0.001102, weight_decay=0.000544),
    'B': dict(hidden_dim=128, out_dim=128, num_heads=4, num_layers=2,
              dropout=0.5241, lr=0.001009, weight_decay=0.000567),
}
P = _PARAMS[OPTION]

# Negative Sampling 
CONTRAST_W   = 0.22871624186804143
DIVERSITY_W  = 0.008352875869117645
BPR_W        = 0.10641806730281829
BRAND_DIV_W  = 0.08587411134475015
BRAND_DIV_M  = 0.4023721409318248
BARLOW_W     = 0.08411677378518012
BARLOW_LAM   = 0.018788018302584773
NEG_RATIO    = 4.000370100031652
HARD_FRAC    = 0.15862681798818878
CAND_MULT    = 8
GRAD_CLIP    = 1.0

HARD_NEG_EVERY = 1

target_edge_type     = ('youtuber', 'collaborate_with', 'brand')
rev_target_edge_type = ('brand', 'partner_with', 'youtuber')
YOUTUBER_NODE_TYPE   = 'youtuber'
BRAND_NODE_TYPE      = 'brand'

USE_MPS_IF_AVAILABLE = False

if torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_MPS_IF_AVAILABLE and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print(f'Option {OPTION} | device={device}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Epochs={MAX_EPOCHS}  Patience={PATIENCE}  Folds={K_FOLDS}')
print(f'Arch: hidden={P["hidden_dim"]} out={P["out_dim"]} heads={P["num_heads"]} layers={P["num_layers"]}')
print(f'Optim: lr={P["lr"]}  wd={P["weight_decay"]}  dropout={P["dropout"]}')
print(f'Negative sampling: ratio={NEG_RATIO:.4f}, hard_frac={HARD_FRAC:.4f}, hard_every={HARD_NEG_EVERY}')


## 1. Load Data

In [ ]:
print(f'Loading {DATA_PATH} ...')
data = torch.load(DATA_PATH, weights_only=False)

in_dims = {nt: data[nt].x.size(-1) for nt in data.node_types}
print(f'in_dims = {in_dims}')

_raw = torch.load(DATA_PATH, weights_only=False)['brand'].x.float()
_ohe = _raw[:, :INDUSTRY_DIM]
industry_labels = _ohe.argmax(dim=1).clone()
industry_labels[_ohe.sum(dim=1) == 0] = -1
print(f'Industry labels: {(industry_labels >= 0).sum().item()} known')

## 2. Model

In [ ]:
class HGTEncoder(nn.Module):
    def __init__(self, metadata, in_dims, hidden_dim, out_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.dropout = dropout
        self.input_lin = nn.ModuleDict(
            {nt: nn.Linear(d, hidden_dim) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([
            HGTConv(hidden_dim, hidden_dim, metadata=metadata, heads=num_heads)
            for _ in range(num_layers)])
        self.output_lin = nn.ModuleDict(
            {nt: nn.Linear(hidden_dim, out_dim) for nt in metadata[0]})

    def forward(self, x_dict, edge_index_dict):
        x_dict = {nt: F.relu(self.input_lin[nt](x.float())) for nt, x in x_dict.items()}
        for conv in self.convs:
            res  = {nt: x.clone() for nt, x in x_dict.items()}
            xout = conv(x_dict, edge_index_dict)
            x_dict = {nt: F.dropout(F.relu(xout[nt] + res[nt]),
                                    p=self.dropout, training=self.training)
                      for nt in xout}
        return {nt: self.output_lin[nt](x) for nt, x in x_dict.items()}


class MLPDecoder(nn.Module):
    def __init__(self, in_dim, dropout):
        super().__init__()
        h = in_dim
        self.mlp = nn.Sequential(
            nn.Linear(in_dim*4, h), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h, h//2),    nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h//2, 1))

    def forward(self, z_src, z_dst, eli):
        s, d = eli
        sv, dv = z_src[s], z_dst[d]
        return self.mlp(torch.cat([sv, dv, sv*dv, (sv-dv).abs()], dim=-1)).squeeze(-1)


class HGTModel(nn.Module):
    def __init__(self, metadata, in_dims, **kw):
        super().__init__()
        self.encoder = HGTEncoder(metadata, in_dims, **kw)
        self.decoder = MLPDecoder(kw['out_dim'], kw['dropout'])

    def forward(self, hdata, tgt):
        z = self.encoder(hdata.x_dict, hdata.edge_index_dict)
        src_t, _, dst_t = tgt
        eli = hdata[tgt].edge_label_index
        return self.decoder(z[src_t], z[dst_t], eli), z

print('Model defined.')

## 3. Loss Functions

In [ ]:
def industry_contrastive_loss(z_brand, ind_labels, num_pairs=256, margin=0.8):
    dev = z_brand.device; n = z_brand.size(0); lbl = ind_labels.to(dev)
    ia = torch.randint(0, n, (num_pairs,), device=dev)
    ib = torch.randint(0, n, (num_pairs,), device=dev)
    valid = (lbl[ia]>=0)&(lbl[ib]>=0)&(ia!=ib)
    if valid.sum()==0: return torch.tensor(0., device=dev)
    ia, ib = ia[valid], ib[valid]
    cos = (F.normalize(z_brand[ia],dim=-1)*F.normalize(z_brand[ib],dim=-1)).sum(-1)
    diff = lbl[ia]!=lbl[ib]
    if diff.sum()==0: return torch.tensor(0., device=dev)
    return F.relu(cos[diff]-(1.-margin)).mean()

def embedding_diversity_loss(z, max_nodes=512, margin=0.60):
    if z is None or z.size(0)<=2: return torch.tensor(0., device=z.device)
    if z.size(0)>max_nodes: z=z[torch.randperm(z.size(0),device=z.device)[:max_nodes]]
    zn=F.normalize(z,p=2,dim=1); sim=zn@zn.T
    mask=~torch.eye(sim.size(0),dtype=torch.bool,device=sim.device)
    return F.relu(sim[mask]-margin).pow(2).mean()

def barlow_twins_loss(z, lambda_off=0.005, max_nodes=1024):
    if z is None or z.size(0)<=2: return torch.tensor(0., device=z.device)
    if z.size(0)>max_nodes: z=z[torch.randperm(z.size(0),device=z.device)[:max_nodes]]
    zn=(z-z.mean(0))/(z.std(0)+1e-8); c=(zn.T@zn)/zn.size(0)
    on=torch.diagonal(c).add_(-1).pow_(2).sum()
    off=c[~torch.eye(c.size(0),dtype=torch.bool,device=c.device)].pow(2).sum()
    return on+lambda_off*off

def bpr_loss(pos_s, neg_s):
    if pos_s.numel()==0 or neg_s.numel()==0: return torch.tensor(0., device=pos_s.device)
    m=max(1,neg_s.numel()//pos_s.numel())
    return -F.logsigmoid(pos_s.repeat_interleave(m)[:neg_s[:pos_s.numel()*m].numel()]
                         -neg_s[:pos_s.numel()*m]).mean()

print('Loss functions defined.')

## 4. Negative Sampling & Train/Eval

In [ ]:
def get_pos_eli(split, tgt):
    """Select only the positive edges from edge_label."""
    eli = split[tgt].edge_label_index
    y   = getattr(split[tgt], 'edge_label', None)
    if y is not None and y.numel() == eli.size(1):
        eli = eli[:, y.bool()]
    return eli


def random_negative_sample(split, full, tgt, pos_eli, n_neg):
    """Perform random negative sampling based on the existing target edges of the full graph."""
    src_t, _, dst_t = tgt
    neg = negative_sampling(
        edge_index=full[tgt].edge_index.to(pos_eli.device),
        num_nodes=(split[src_t].num_nodes, split[dst_t].num_nodes),
        num_neg_samples=n_neg,
        method='sparse'
    )
    if neg.numel() == 0:
        raise RuntimeError('negative_sampling returned empty tensor.')
    return neg[:, :n_neg]


@torch.no_grad()
def sample_mixed_negative(model, split, full, tgt, pos_eli, epoch=None):
    """
    Mixed negative sampling:
    - Each positive edge is paired with NEG_RATIO negative edges.
    - If HARD_FRAC > 0 and it is a HARD_NEG_EVERY epoch, pick the
      highest-scoring candidates from the candidate negatives as hard negatives.
    - The remaining negative edges use random negatives.
    """
    dev = pos_eli.device
    src_t, _, dst_t = tgt

    n_neg = max(1, int(math.ceil(pos_eli.size(1) * NEG_RATIO)))
    use_hard = (
        HARD_FRAC > 0
        and model is not None
        and (epoch is None or epoch % HARD_NEG_EVERY == 0)
    )

    if not use_hard:
        return random_negative_sample(split, full, tgt, pos_eli, n_neg)

    n_cand = max(n_neg, int(n_neg * CAND_MULT))
    cand = negative_sampling(
        edge_index=full[tgt].edge_index.to(dev),
        num_nodes=(split[src_t].num_nodes, split[dst_t].num_nodes),
        num_neg_samples=n_cand,
        method='sparse'
    )
    if cand.numel() == 0:
        return random_negative_sample(split, full, tgt, pos_eli, n_neg)

    model.eval()
    z = model.encoder(split.x_dict, split.edge_index_dict)
    scores = torch.sigmoid(model.decoder(z[src_t], z[dst_t], cand).view(-1))

    n_hard = min(int(round(n_neg * HARD_FRAC)), cand.size(1))
    n_rand = n_neg - n_hard

    hard_neg = cand[:, :0]
    hard_idx = torch.empty(0, dtype=torch.long, device=dev)

    if n_hard > 0:
        hard_idx = torch.topk(scores, k=n_hard).indices
        hard_neg = cand[:, hard_idx]

    if n_rand > 0:
        mask = torch.ones(cand.size(1), dtype=torch.bool, device=dev)
        if n_hard > 0:
            mask[hard_idx] = False
        remain = torch.where(mask)[0]

        if remain.numel() >= n_rand:
            rand_idx = remain[torch.randperm(remain.numel(), device=dev)[:n_rand]]
            rand_neg = cand[:, rand_idx]
        else:
            rand_neg = random_negative_sample(split, full, tgt, pos_eli, n_rand)

        neg_ei = torch.cat([hard_neg, rand_neg], dim=1)
    else:
        neg_ei = hard_neg

    return neg_ei[:, :n_neg]


def train_epoch(model, train_data, full_data, tgt, optimizer, epoch):
    """Return (total_loss, bce_loss), so they can be plotted separately."""
    model.train()

    pos_eli = get_pos_eli(train_data, tgt)
    neg_eli = sample_mixed_negative(model, train_data, full_data, tgt, pos_eli, epoch=epoch)

    eli = torch.cat([pos_eli, neg_eli], dim=1)
    y = torch.cat([
        torch.ones(pos_eli.size(1), device=eli.device),
        torch.zeros(neg_eli.size(1), device=eli.device)
    ])

    src_t, _, dst_t = tgt

    optimizer.zero_grad()
    z = model.encoder(train_data.x_dict, train_data.edge_index_dict)
    logits = model.decoder(z[src_t], z[dst_t], eli).view(-1)

    bce = nn.BCEWithLogitsLoss()(logits, y)
    bpr = bpr_loss(logits[:pos_eli.size(1)], logits[pos_eli.size(1):])

    c_l = industry_contrastive_loss(z[BRAND_NODE_TYPE], industry_labels.to(eli.device)) \
          if CONTRAST_W > 0 else torch.tensor(0., device=eli.device)
    ydiv = embedding_diversity_loss(z[YOUTUBER_NODE_TYPE]) \
           if DIVERSITY_W > 0 else torch.tensor(0., device=eli.device)
    bdiv = embedding_diversity_loss(z[BRAND_NODE_TYPE], max_nodes=512, margin=BRAND_DIV_M) \
           if BRAND_DIV_W > 0 else torch.tensor(0., device=eli.device)
    bt = barlow_twins_loss(z[BRAND_NODE_TYPE], lambda_off=BARLOW_LAM) \
         if BARLOW_W > 0 else torch.tensor(0., device=eli.device)

    total = (
        bce
        + BPR_W * bpr
        + CONTRAST_W * c_l
        + DIVERSITY_W * ydiv
        + BRAND_DIV_W * bdiv
        + BARLOW_W * bt
    )

    total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

    return float(total.detach().cpu()), float(bce.detach().cpu())


@torch.no_grad()
def evaluate_split(model, split_data, tgt, return_loss=True):
    """Return the BCE loss, AUC, and AP for the split."""
    model.eval()
    logits, _ = model(split_data, tgt)
    logits = logits.view(-1)
    y = split_data[tgt].edge_label.float().view(-1)

    y_np = y.detach().cpu().numpy()
    s_np = logits.detach().cpu().numpy()

    auc = roc_auc_score(y_np, s_np)
    ap = average_precision_score(y_np, s_np)

    if return_loss:
        bce = nn.BCEWithLogitsLoss()(logits, y).item()
        return bce, auc, ap
    return auc, ap


@torch.no_grad()
def compute_embedding_metrics(model, hdata, node_type=BRAND_NODE_TYPE, max_nodes=None):
    """
    Compute embedding collapse metrics:
    - eff_rank: effective rank; higher means more dimensions are being used.
    - k95: number of dimensions needed to explain 95% of the energy; higher
      means the dimensions are more spread out.
    - mean_cos: off-diagonal mean absolute cosine similarity; lower means
      node embeddings are less similar to each other.
    """
    model.eval()
    z = model.encoder(hdata.x_dict, hdata.edge_index_dict)[node_type].detach()

    if max_nodes is not None and z.size(0) > max_nodes:
        idx = torch.randperm(z.size(0), device=z.device)[:max_nodes]
        z = z[idx]

    z = z.float()
    z_centered = z - z.mean(dim=0, keepdim=True)

    s = torch.linalg.svdvals(z_centered)
    s = torch.clamp(s, min=1e-12)
    p = s / s.sum()
    eff_rank = torch.exp(-(p * torch.log(p)).sum()).item()

    energy = s.pow(2)
    cum_energy = torch.cumsum(energy, dim=0) / energy.sum()
    k95 = int((cum_energy >= 0.95).nonzero(as_tuple=False)[0].item() + 1)

    zn = F.normalize(z, p=2, dim=1)
    sim = zn @ zn.T
    mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
    mean_cos = sim[mask].abs().mean().item()

    return {
        'eff_rank': float(eff_rank),
        'k95': int(k95),
        'mean_cos': float(mean_cos),
    }


def edge_pairs_tensor_to_set(edge_index):
    """Convert edge_index into a set, used for the leakage sanity check."""
    ei = edge_index.detach().cpu()
    return set(zip(ei[0].tolist(), ei[1].tolist()))


def leakage_sanity_check(train_data, val_data, test_data, tgt):
    """
    Check whether validation/test positive target edges are still present in
    the train message-passing graph.
    If RandomLinkSplit + disjoint_train_ratio is configured correctly, this
    should theoretically be close to 0.
    """
    train_mp = edge_pairs_tensor_to_set(train_data[tgt].edge_index)
    val_pos  = edge_pairs_tensor_to_set(get_pos_eli(val_data, tgt))
    test_pos = edge_pairs_tensor_to_set(get_pos_eli(test_data, tgt))

    val_overlap = len(train_mp.intersection(val_pos))
    test_overlap = len(train_mp.intersection(test_pos))

    return {
        'train_message_edges': len(train_mp),
        'val_pos_edges': len(val_pos),
        'test_pos_edges': len(test_pos),
        'val_overlap_in_train_mp': val_overlap,
        'test_overlap_in_train_mp': test_overlap,
    }


print('Negative sampling, evaluation, embedding metrics, and leakage checks defined.')


## 5. Main Training Loop

In [ ]:
def train_fold(fold_idx, train_data, val_data, test_data, full_data, verbose_step=10):
    """
    Train a single fold.

    Early stopping:
    1. Primarily based on Val AUC.
    2. If epoch > VAL_LOSS_PROTECTION_EPOCH and Val BCE has clearly deteriorated, stop early.
    3. Each fold saves its best checkpoint.
    """
    model = HGTModel(
        metadata=data.metadata(),
        in_dims=in_dims,
        hidden_dim=P['hidden_dim'],
        out_dim=P['out_dim'],
        num_heads=P['num_heads'],
        num_layers=P['num_layers'],
        dropout=P['dropout']
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=P['lr'],
        weight_decay=P['weight_decay']
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        patience=8,
        factor=0.5
    )

    history = {
        'epoch': [],
        'total_train_loss': [],
        'train_bce_loss': [],
        'val_bce_loss': [],
        'val_auc': [],
        'val_ap': [],
        'lr': [],
    }

    best_val_auc = -float('inf')
    best_val_ap  = -float('inf')
    best_val_bce = float('inf')
    best_epoch = 0
    best_state = None
    no_improve = 0
    stop_reason = 'max_epochs'

    print('  ' + '=' * 60)
    print(f'  Fold {fold_idx} / {K_FOLDS}')
    print('  ' + '=' * 60)

    start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        total_loss, bce_loss = train_epoch(
            model, train_data, full_data, target_edge_type, optimizer, epoch
        )

        val_bce, val_auc, val_ap = evaluate_split(model, val_data, target_edge_type)

        current_lr = optimizer.param_groups[0]['lr']

        history['epoch'].append(epoch)
        history['total_train_loss'].append(total_loss)
        history['train_bce_loss'].append(bce_loss)
        history['val_bce_loss'].append(val_bce)
        history['val_auc'].append(val_auc)
        history['val_ap'].append(val_ap)
        history['lr'].append(current_lr)

        scheduler.step(val_auc)

        # Early stopping 
        auc_improved = val_auc > best_val_auc + VAL_AUC_MIN_DELTA

        if auc_improved:
            best_val_auc = val_auc
            best_val_ap = val_ap
            best_val_bce = val_bce
            best_epoch = epoch
            no_improve = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if val_bce < best_val_bce:
                best_val_bce = val_bce

        if epoch % verbose_step == 0 or epoch == 1:
            print(
                f'  E{epoch:03d} | '
                f'TotalLoss={total_loss:.4f} BCE={bce_loss:.4f} | '
                f'ValBCE={val_bce:.4f} ValAUC={val_auc:.4f} ValAP={val_ap:.4f} | '
                f'LR={current_lr:.2e} | NoImp={no_improve}/{PATIENCE}'
            )

        if no_improve >= PATIENCE:
            stop_reason = f'patience({PATIENCE})'
            print(f'  ⏹ Early stop @ E{epoch} [patience], Best @ E{best_epoch}')
            break

        if (
            epoch > VAL_LOSS_PROTECTION_EPOCH
            and val_bce > best_val_bce + VAL_LOSS_PROTECTION_DELTA
        ):
            stop_reason = f'val_loss_protection(+{VAL_LOSS_PROTECTION_DELTA})'
            print(f'  ⏹ Early stop @ E{epoch} [val loss deterioration], Best @ E{best_epoch}')
            break

    elapsed_min = (time.time() - start_time) / 60

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    test_bce, test_auc, test_ap = evaluate_split(model, test_data, target_edge_type)

    # Compute brand embedding metrics using the train_data message-passing graph.
    brand_metrics = compute_embedding_metrics(model, train_data, BRAND_NODE_TYPE)

    checkpoint_path = os.path.join(
        OUTPUT_DIR,
        f'hgt_final_option{OPTION}_fold{fold_idx}_best.pt'
    )

    torch.save({
        'fold': fold_idx,
        'option': OPTION,
        'model_state_dict': best_state,
        'params': P,
        'config': {
            'max_epochs': MAX_EPOCHS,
            'patience': PATIENCE,
            'val_auc_min_delta': VAL_AUC_MIN_DELTA,
            'val_loss_protection_epoch': VAL_LOSS_PROTECTION_EPOCH,
            'val_loss_protection_delta': VAL_LOSS_PROTECTION_DELTA,
            'disjoint_ratio': DISJOINT_RATIO,
            'neg_ratio': NEG_RATIO,
            'hard_frac': HARD_FRAC,
            'hard_neg_every': HARD_NEG_EVERY,
            'contrast_w': CONTRAST_W,
            'diversity_w': DIVERSITY_W,
            'bpr_w': BPR_W,
            'brand_div_w': BRAND_DIV_W,
            'barlow_w': BARLOW_W,
        },
        'metrics': {
            'best_epoch': best_epoch,
            'best_val_auc': best_val_auc,
            'best_val_ap': best_val_ap,
            'test_bce': test_bce,
            'test_auc': test_auc,
            'test_ap': test_ap,
            **brand_metrics,
            'stop_reason': stop_reason,
            'elapsed_min': elapsed_min,
        },
        'in_dims': in_dims,
        'target_edge_type': target_edge_type,
        'rev_target_edge_type': rev_target_edge_type,
    }, checkpoint_path)

    print(
        f'  Fold {fold_idx} | BestValAUC={best_val_auc:.4f} @ E{best_epoch} | '
        f'TestAUC={test_auc:.4f} | TestAP={test_ap:.4f} | '
        f'eff_rank={brand_metrics["eff_rank"]:.2f} | mean_cos={brand_metrics["mean_cos"]:.4f} | '
        f'Stop={stop_reason}'
    )
    print(f'  Saved checkpoint: {checkpoint_path}')

    fold_result = {
        'fold': fold_idx,
        'best_epoch': best_epoch,
        'best_val_auc': best_val_auc,
        'best_val_ap': best_val_ap,
        'test_bce': test_bce,
        'test_auc': test_auc,
        'test_ap': test_ap,
        'stop_reason': stop_reason,
        'elapsed_min': elapsed_min,
        **brand_metrics,
        'checkpoint_path': checkpoint_path,
    }

    return history, fold_result


print('train_fold() defined.')


## 6. Run K-Fold Training

In [ ]:
all_histories = []
fold_results = []
leakage_reports = []

for fold in range(1, K_FOLDS + 1):
    fold_seed = SEED + fold
    set_seed(fold_seed)

    transform = T.RandomLinkSplit(
        num_val=0.1,
        num_test=0.1,
        is_undirected=False,
        disjoint_train_ratio=DISJOINT_RATIO,
        edge_types=[target_edge_type],
        rev_edge_types=[rev_target_edge_type],
        add_negative_train_samples=False,
        neg_sampling_ratio=NEG_RATIO,
    )

    train_data, val_data, test_data = transform(data)

    leak_report = leakage_sanity_check(train_data, val_data, test_data, target_edge_type)
    leak_report['fold'] = fold
    leakage_reports.append(leak_report)

    print(f'\nLeakage check Fold {fold}: '
          f'val_overlap={leak_report["val_overlap_in_train_mp"]}, '
          f'test_overlap={leak_report["test_overlap_in_train_mp"]}')

    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)
    full_data  = data.to(device)

    hist, fold_result = train_fold(
        fold,
        train_data,
        val_data,
        test_data,
        full_data,
        verbose_step=10
    )

    all_histories.append(hist)
    fold_results.append(fold_result)

    del train_data, val_data, test_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fold_df = pd.DataFrame(fold_results)
leakage_df = pd.DataFrame(leakage_reports)

print()
print('=' * 60)
print('Cross-Validation Summary')
print('=' * 60)

display_cols = [
    'fold', 'best_epoch', 'best_val_auc', 'best_val_ap',
    'test_auc', 'test_ap', 'eff_rank', 'k95', 'mean_cos', 'stop_reason'
]
print(fold_df[display_cols].to_string(index=False))

print()
print(f'Mean TestAUC : {fold_df["test_auc"].mean():.4f} ± {fold_df["test_auc"].std(ddof=0):.4f}')
print(f'Mean TestAP  : {fold_df["test_ap"].mean():.4f} ± {fold_df["test_ap"].std(ddof=0):.4f}')
print(f'Mean BestEpoch: {fold_df["best_epoch"].mean():.1f}')
print(f'Mean eff_rank : {fold_df["eff_rank"].mean():.2f}')
print(f'Mean mean_cos : {fold_df["mean_cos"].mean():.4f}')

print()
print('=' * 60)
print('Leakage Check Summary')
print('=' * 60)
print(leakage_df.to_string(index=False))

# Save intermediate results
fold_csv = os.path.join(OUTPUT_DIR, f'hgt_final_option{OPTION}_fold_summary.csv')
leak_csv = os.path.join(OUTPUT_DIR, f'hgt_final_option{OPTION}_leakage_check.csv')
fold_df.to_csv(fold_csv, index=False, encoding='utf-8-sig')
leakage_df.to_csv(leak_csv, index=False, encoding='utf-8-sig')
print(f'\nSaved: {fold_csv}')
print(f'Saved: {leak_csv}')


## 7. Four Charts: Split Loss + AUC + AP

In [ ]:
def pad_and_stack(all_hist, key):
    """Fold lengths may differ; pad with NaN before computing mean/std."""
    seqs = [np.array(h[key], dtype=float) for h in all_hist]
    maxlen = max(len(s) for s in seqs)
    padded = np.full((len(seqs), maxlen), np.nan)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    return padded


def plot_mean_std(ax, mat, label, linestyle='-'):
    """Plot mean ± std, ignoring NaN."""
    mu = np.nanmean(mat, axis=0)
    std = np.nanstd(mat, axis=0)
    x = np.arange(1, len(mu) + 1)
    ax.plot(x, mu, label=label, linestyle=linestyle, linewidth=2)
    ax.fill_between(x, mu - std, mu + std, alpha=0.15)


total_train = pad_and_stack(all_histories, 'total_train_loss')
train_bce   = pad_and_stack(all_histories, 'train_bce_loss')
val_bce     = pad_and_stack(all_histories, 'val_bce_loss')
val_auc_mat = pad_and_stack(all_histories, 'val_auc')
val_ap_mat  = pad_and_stack(all_histories, 'val_ap')

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
plot_mean_std(ax1, train_bce, 'Train Loss')
plot_mean_std(ax1, val_bce, 'Validation Loss')
ax1.set_title('Train vs Validation Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
plot_mean_std(ax2, total_train, 'Total Train Loss')
ax2.set_title('Total Train Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Total Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 0])
plot_mean_std(ax3, val_auc_mat, 'Validation AUC')
peak_auc = np.nanmax(np.nanmean(val_auc_mat, axis=0))
ax3.axhline(peak_auc, linestyle='--', linewidth=1, alpha=0.6, label='Peak Mean AUC')
ax3.set_title('Validation AUC', fontsize=13, fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('AUC')
ax3.set_ylim(0.5, 1.0)
ax3.legend()
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(gs[1, 1])
plot_mean_std(ax4, val_ap_mat, 'Validation AP')
ax4.set_title('Validation Average Precision', fontsize=13, fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('AP')
ax4.set_ylim(0.0, 1.0)
ax4.legend()
ax4.grid(True, alpha=0.3)

fig.suptitle(
    f'HGT Final Training — Option {OPTION} | '
    f'{K_FOLDS}-Fold CV | '
    f'Mean Test AUC={fold_df["test_auc"].mean():.4f}±{fold_df["test_auc"].std(ddof=0):.4f} | '
    f'Mean Test AP={fold_df["test_ap"].mean():.4f}±{fold_df["test_ap"].std(ddof=0):.4f}',
    fontsize=13,
    y=1.01
)

plot_path = os.path.join(OUTPUT_DIR, f'hgt_final_option{OPTION}_training_curves.png')
plt.savefig(plot_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')


## 8. Save Results

In [ ]:
summary_result = {
    'option': OPTION,
    'params': P,
    'fold_results': fold_df.to_dict(orient='records'),
    'leakage_reports': leakage_df.to_dict(orient='records'),
    'summary': {
        'mean_test_auc': float(fold_df['test_auc'].mean()),
        'std_test_auc': float(fold_df['test_auc'].std(ddof=0)),
        'mean_test_ap': float(fold_df['test_ap'].mean()),
        'std_test_ap': float(fold_df['test_ap'].std(ddof=0)),
        'mean_best_epoch': float(fold_df['best_epoch'].mean()),
        'mean_eff_rank': float(fold_df['eff_rank'].mean()),
        'mean_k95': float(fold_df['k95'].mean()),
        'mean_mean_cos': float(fold_df['mean_cos'].mean()),
    },
    'config': {
        'data_path': DATA_PATH,
        'k_folds': K_FOLDS,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'val_auc_min_delta': VAL_AUC_MIN_DELTA,
        'val_loss_protection_epoch': VAL_LOSS_PROTECTION_EPOCH,
        'val_loss_protection_delta': VAL_LOSS_PROTECTION_DELTA,
        'early_stop_metric': 'validation_auc_with_validation_loss_protection',
        'disjoint_ratio': DISJOINT_RATIO,
        'apply_pca_whitening': APPLY_PCA_WHITENING,
        'variance_threshold': VARIANCE_THRESHOLD,
        'negative_sampling_ratio': NEG_RATIO,
        'hard_fraction': HARD_FRAC,
        'hard_negative_every': HARD_NEG_EVERY,
        'grad_clip': GRAD_CLIP,
        'loss_weights': {
            'contrast_w': CONTRAST_W,
            'diversity_w': DIVERSITY_W,
            'bpr_w': BPR_W,
            'brand_div_w': BRAND_DIV_W,
            'brand_div_margin': BRAND_DIV_M,
            'barlow_w': BARLOW_W,
            'barlow_lambda': BARLOW_LAM,
        },
        'target_edge_type': target_edge_type,
        'rev_target_edge_type': rev_target_edge_type,
        'in_dims': in_dims,
    }
}

json_path = os.path.join(OUTPUT_DIR, f'hgt_final_option{OPTION}_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(summary_result, f, ensure_ascii=False, indent=2)

best_row = fold_df.sort_values(['best_val_auc', 'best_val_ap'], ascending=False).iloc[0]
best_checkpoint_path = best_row['checkpoint_path']
best_pointer_path = os.path.join(OUTPUT_DIR, f'hgt_final_option{OPTION}_best_checkpoint_pointer.json')

with open(best_pointer_path, 'w', encoding='utf-8') as f:
    json.dump({
        'selection_rule': 'highest best_val_auc, then highest best_val_ap',
        'selected_fold': int(best_row['fold']),
        'best_checkpoint_path': best_checkpoint_path,
        'best_val_auc': float(best_row['best_val_auc']),
        'best_val_ap': float(best_row['best_val_ap']),
        'test_auc_of_selected_fold': float(best_row['test_auc']),
        'test_ap_of_selected_fold': float(best_row['test_ap']),
    }, f, ensure_ascii=False, indent=2)

print(f'Saved: {json_path}')
print(f'Saved: {best_pointer_path}')
print()
print(f'Mean TestAUC = {summary_result["summary"]["mean_test_auc"]:.4f} ± {summary_result["summary"]["std_test_auc"]:.4f}')
print(f'Mean TestAP  = {summary_result["summary"]["mean_test_ap"]:.4f} ± {summary_result["summary"]["std_test_ap"]:.4f}')
print(f'Mean eff_rank = {summary_result["summary"]["mean_eff_rank"]:.2f}')
print(f'Mean mean_cos = {summary_result["summary"]["mean_mean_cos"]:.4f}')
print()
print('Best checkpoint for later recommendation/explainer analysis:')
print(best_checkpoint_path)
